### Setup Prompt Governor

In [ ]:
import os
import components.prompt_governor as pg
from importlib import reload

pg = reload(pg)

ActionSchema = pg.ActionSchema
ActionGovernor = pg.ActionGovernor
ElementSchema = pg.ElementSchema
ElementGovernor = pg.ElementGovernor
StateSchema = pg.StateSchema
StateGovernor = pg.StateGovernor
BBoxSchema = pg.BBoxRefinementSchema
BBoxGovernor = pg.BBoxRefinementGovernor
ClickSchema = pg.ClickPointSchema
ClickGovernor = pg.ClickPointGovernor

action_schema = ActionSchema(
    enable_disallowed_filter=True,
    disallowed_match_case_sensitive=True,
)

action_governor = ActionGovernor(
    schema=action_schema,
    enable_autofix=True,
    enforce_confidence_sort=True,
)

element_schema = ElementSchema(
    relevance_min=1,
    relevance_max=10,
    enforce_value_null_when_not_type=True,
    enforce_value_nonnull_when_type=True,
)

element_governor = ElementGovernor(
    schema=element_schema,
    enable_autofix=True,
)

state_schema = StateSchema(
    state_min=1,
    state_max=6,
)

state_governor = StateGovernor(
    schema=state_schema,
    enable_autofix=True,
)

bbox_schema = BBoxSchema()

bbox_governor = BBoxGovernor(
    schema=bbox_schema,
    enable_autofix=True,
)

click_schema = ClickSchema()

click_governor = ClickGovernor(
    schema=click_schema,
    enable_autofix=True,
)

### Module

In [ ]:
from pathlib import Path
import json

import components.agent as agent
import components.credential_fetcher as cf
from importlib import reload
reload(cf)
reload(agent)

import components.element_utils
reload(components.element_utils)
from components.element_utils import resolve_action_element

In [ ]:
import logging
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

@dataclass
class ExecContext:
    vlm: Any
    tree: Any
    agent: Any
    driver: Any
    current_node: int
    image_path: str
    domain: str
    state: Optional[int] = None

    # debugging / tracing
    debug: Dict[str, Any] = field(default_factory=dict)
    scrolled: List[str] = field(default_factory=list)
    logger: Optional[logging.Logger] = None


In [ ]:
@dataclass
class StateResult:
    decision: str
    state: int
    keywords: list
    
def run_state_module(ctx: ExecContext, image_path=None) -> StateResult:
    
    if image_path is None:
        image_path = ctx.image_path
    
    raw_resp, resp, prompt = ctx.agent.state_identifier(ctx.vlm, ctx.tree, image_path, ctx.current_node)

    ctx.debug["state_prompt"] = prompt
    ctx.debug["state_resp"] = resp

    # with open(log_path, "a", encoding="utf-8") as f:
    
    ctx.logger.info(
        "[INFO] STATE IDENTIFICATION raw response:\n%s\n",
        raw_resp
    )

    result = state_governor.govern(resp).fixed_output
    
    ctx.logger.info(
        "%s\n"
        "[NEXT STATE IDENTIFICATION]\n"
        "node=%s\n"
        "image=%s\n"
        "%s\n"
        "%s\n\n",
        "-" * 60,
        ctx.current_node,
        ctx.image_path,
        resp,
        "-" * 60,
    )

    decision = result['decision']
    state = result['state']
    _keywords = result['search_keywords']
    
    keywords = [
        kw for kw in _keywords
        if not any(x in kw["text"].lower() for x in ("gmail", "outlook"))
    ]
    
    ctx.logger.info("[INFO] search_keywords: %s\n", str(keywords))
    
    return StateResult(
        decision=decision,
        state=state,
        keywords=keywords
    )

In [ ]:
import components.element_utils as eu
reload(eu)

In [ ]:
from urllib.parse import urljoin
from components.element_utils import resolve_action_element_from_item

@dataclass
class ActionResult:
    next_node: int
    actions: list
    items: list

def run_action_module(
    ctx: ExecContext,
    keywords,
    action_governor
    # log_path: str,
) -> ActionResult:
    """
    - action generation
    - governor
    - tree expansion
    - select next current_node
    """

    tree = ctx.tree
    current_node = ctx.current_node
    
    items = eu.collect_keyword_elements(ctx.driver, keywords)
    items_str = eu.format_nav_elements_for_prompt(ctx.driver, items, keywords)
    if items_str == '':
        items_str = "NULL"
    
    ctx.logger.info("[INFO] items_str:\n%s\n", items_str)
    
    viewport_info = eu.get_viewport_info(ctx.driver)
    # 1. action generation
    raw_resp, action_resp, disallowed, prompt = ctx.agent.action_generator(
        ctx.vlm, tree, ctx.image_path, ctx.state, current_node, items_str, viewport_info
    )
    ctx.logger.info(
        "[INFO] ACTION GENERATION raw response:\n%s\n"
        "[INFO] disallowed actions:\n%s\n",
        raw_resp,
        json.dumps(disallowed, ensure_ascii=False, indent=2)
    )

    ctx.debug["action_resp"] = action_resp
    ctx.debug["disallowed_actions"] = disallowed
    ctx.debug["action_prompt"] = prompt

    # 2. governor
    result = action_governor.govern(
        action_resp, disallowed_actions=disallowed
    )
    
    ctx.debug["action_result"] = result

    if not result.ok:
        raise RuntimeError(
            "Governor rejected output:\n"
            + "\n".join(f"- {v.code}: {v.message}" for v in result.violations)
        )

    actions = result.fixed_output

    # 3. logging
    # with open(log_path, "a", encoding="utf-8") as f:
    ctx.logger.info(
        "%s\n"
        "[ACTION GENERATION]\n"
        "state=%s node=%s\n"
        "image=%s\n"
        "%s\n"
        "%s\n\n",
        "-" * 60,
        ctx.state,
        ctx.current_node,
        ctx.image_path,
        json.dumps(actions, ensure_ascii=False, indent=2),
        "-" * 60,
    )
    
    if ctx.driver.current_url in ctx.scrolled:
        prune_scroll = True
    else:
        prune_scroll = False

    tree.mark_expanded(current_node)
    
    for r in actions:
        nid = tree.new_node_id()
        tree.add_child(
            node_id=nid,
            parent_id=current_node,
            action=r["action"],
            confidence_score=r["confidence"],
            state=ctx.state,
        )
        if r["confidence"] < 6:
            tree.prune_node(nid, "low confidence")
            ctx.logger.info(
                "[INFO] pruned node%d due to low confidence (%d)\n",
                nid,
                r["confidence"]
            )
        if prune_scroll and r["action"]["action_type"]=="scroll":
            tree.prune_node(nid, "scroll already reached bottom")
            ctx.logger.info(
                "[INFO] pruned node%d - scroll already reached bottom. no more scrolling is needed.\n",
                nid
            )
        
    parent = tree.nodes[current_node]
    candidates = [
        tree.nodes[cid]
        for cid in parent.children
        if not tree.nodes[cid].pruned
    ]

    if not candidates:
        ctx.logger.info(
            "[INFO] No child generated. "
        )
        
        return ActionResult(
            next_node=None,
            actions=None,
            items = None
        )

    next_node = max(candidates, key=lambda n: n.confidence_score).id
    ctx.logger.info(
        "[INFO] next node is %d\n",
        next_node
    )
    
    equiv_nodes = tree.find_equiv_node_ids(next_node)
    
    for en in equiv_nodes:
        tree.prune_node(en, "equiv node expanded")
        ctx.logger.info(
            "[INFO] pruned node%d: equiv node expanded\n",
            en
        )
    tree.mark_expanded(next_node)

    ctx.debug["action_candidates"] = [n.id for n in candidates]

    return ActionResult(
                next_node=next_node,
                actions=actions,
                items=items
            )

In [ ]:
@dataclass
class ElementResult:
    selected_index: int
    relevance: int
    value: Optional[str]
    filtered_elements: list
    raw_htmls: list
    quadrant_bbox: list
    
def run_element_module(
    ctx: ExecContext,
    bbox_dir,
    element_governor,
    CHUNK_SIZE,
    # log_path: str
) -> ElementResult:

    node = ctx.tree.nodes[ctx.current_node]

    raw_resps, resp, cand_resps, all_hits, filtered, sanitized, raw_htmls, quadrant_bbox, prompt = ctx.agent.element_selector(
        ctx.vlm, ctx.tree, ctx.driver, bbox_dir, ctx.current_node, CHUNK_SIZE
    )
    
    #         "[INFO] No candidate elements detected\n"
        
    ctx.logger.info(
        "[INFO] ELEMENT SELECTION raw response:\n%s\n",
        "\n".join([str(r) for r in raw_resps])
    )

    ctx.debug["element_resp"] = resp
    ctx.debug["filtered_elements"] = filtered
    ctx.debug["all_hits"] = all_hits
    ctx.debug['sanitized'] = sanitized
    ctx.debug["raw_htmls"] = raw_htmls
    ctx.debug["element_prompt"] = prompt

    result = element_governor.govern(resp, proposed_action=node.action)
    
    ctx.debug["element_result"] = result

    if not result.ok:
        raise RuntimeError(
            "Governor rejected output:\n"
            + "\n".join(f"- {v.code}: {v.message}" for v in result.violations)
        )

    obj = result.fixed_output
    
    if len(sanitized) == 0:
        selected_index = -1
        sanitized_str = "no elements"
    else:
        selected_index = obj["selected"]["index"]
        sanitized_str = sanitized[obj["selected"]["index"]-1]
        
    ctx.logger.info(
        "%s\n"
        "[ELEMENT SELECTION]\n"
        "state=%s node=%s\n"
        # "image=%s\n"
        "%s\n"
        "object=%s\n"
        "%s\n\n",
        "-" * 60,
        ctx.state,
        ctx.current_node,
        sanitized_str,
        json.dumps(obj, ensure_ascii=False, indent=2),
        "-" * 60,
    )

    return ElementResult(
        selected_index=selected_index,
        relevance = obj["selected"]["relevance"],
        value=obj.get("value"),
        filtered_elements=filtered,
        raw_htmls = raw_htmls,
        quadrant_bbox = quadrant_bbox
    )


In [ ]:
import components.element_utils as eu
reload(eu)

@dataclass
class RefineBBoxResult:
    matched: bool
    need_prune: bool
    bbox_rel: list
    bbox_view: list
    click_point_rel: list
    click_point_view: list

def run_refine_bbox_module(
    ctx,
    element_bbox,
    quadrant_bbox,
    action_str,
    crop_img_path,
    padding = 10
):
    eu.highlight_bbox(ctx.driver, element_bbox)
    
    eu.save_window_bbox_png(ctx.driver, quadrant_bbox, crop_img_path + ".png")
        
    with open("./prompt/4_BBoxRefinement_system.md", "r", encoding="utf-8") as f:
        system_prompt = f.read()

    with open("./prompt/4_BBoxRefinement_user.md", "r", encoding="utf-8") as f:
        user_prompt_template = f.read()

    crop_img = Image.open(crop_img_path + ".png")
    width, height = crop_img.size
    image_size = {"width": width, "height": height}

    user_prompt = (
        user_prompt_template
        .replace("<<IMAGE_SIZE>>", str(image_size))
        .replace("<<ACTION>>", action_str)
    )

    bbox_rel = eu.convert_bbox_to_crop_space(element_bbox, quadrant_bbox)
    user_prompt = user_prompt.replace("<<BBOX>>", '')

    ctx.logger.info("[INFO] bbox: %s, bbox_rel: %s, quadrant_bbox: %s\n", element_bbox, bbox_rel, quadrant_bbox)
    
    ctx.debug['refine_prompt'] = system_prompt+'\n'+user_prompt
    
    raw_resp, resp = ctx.agent.call_vlm(ctx.vlm, user_prompt=user_prompt, image_path=crop_img_path + ".png", system_prompt=system_prompt)

    ctx.logger.info(
        "[INFO] BBOX REFINEMENT raw response:\n%s\n",
        raw_resp
    )

    result = bbox_governor.govern(resp).fixed_output
    
    ctx.logger.info(
        "%s\n[BBOX REFINEMENT]\nnode=%s\nimage=%s\n%s\n%s\n",
        "-" * 60,
        ctx.current_node,
        crop_img_path + ".png",
        result,
        "-" * 60,
    )

    eu.clear_all_visual_debug(ctx.driver)
    if result["match"] == "yes":
        #         verif_json["click_point"],
        #         quadrant_bbox
        return RefineBBoxResult(matched=True, need_prune=False,
                                bbox_rel=None, bbox_view=None,
                                click_point_rel=None,  click_point_view=None)
    else:
        
        if result["bbox"] is not None:
            x1, y1, x2, y2 = result["bbox"]
            bbox_rel = [
                x1 - padding,
                y1 - padding,
                x2 + padding,
                y2 + padding,
            ]
            bbox_view = eu.convert_crop_bbox_to_viewport(result["bbox"], quadrant_bbox)
        else:
            bbox_rel = None
            bbox_view = None
        
        return RefineBBoxResult(matched=False, need_prune=False,
                                bbox_rel=bbox_rel, bbox_view=bbox_view,
                                click_point_rel=None, click_point_view=None)

def run_bbox_adjustment_loop(
    ctx,
    node,
    # bbox_rel,
    # bbox_view,
    quadrant_bbox,
    crop_img_path,
    action_str,
    value,
    rank,
    LOOP= 3,
    padding = 10
):
    adjust_idx = 1

    while adjust_idx <= LOOP:

        adjust_img_path = f"{crop_img_path}-{adjust_idx}.png"

        ctx.logger.info("[INFO] quadrant_bbox: %s", str(quadrant_bbox))
        if adjust_idx == 1:
            x1, y1, x2, y2 = node.action["bbox"]
            bbox_view = [
                x1 - padding,
                y1 - padding,
                x2 + padding,
                y2 + padding,
            ]
            quadrant_bbox = eu.ret_quadrant_bbox(ctx.driver, bbox_view)
            bbox_rel = eu.convert_bbox_to_crop_space(bbox_view, quadrant_bbox)
        else:
            bbox_view = eu.convert_crop_bbox_to_viewport(bbox_rel, quadrant_bbox)
            quadrant_bbox = eu.ret_quadrant_bbox(ctx.driver, bbox_view)
            
        ctx.logger.info("[INFO] quadrant_bbox: %s\n", str(quadrant_bbox))
        ctx.debug['quadrant_bbox'] = quadrant_bbox

        eu.highlight_bbox(ctx.driver, bbox_view)
        eu.save_window_bbox_png(ctx.driver, quadrant_bbox, adjust_img_path)

        adjust_img = Image.open(adjust_img_path)
        width, height = adjust_img.size
        image_size = {"width": width, "height": height}

        with open("./prompt/4_BBoxRefinement_system.md", "r", encoding="utf-8") as f:
            system_prompt = f.read()

        with open("./prompt/4_BBoxRefinement_user.md", "r", encoding="utf-8") as f:
            user_prompt_template = f.read()

        clamped_bbox = eu.clamp_bbox_to_image(bbox_rel, image_size)

        user_prompt = (
            user_prompt_template
            .replace("<<IMAGE_SIZE>>", str(image_size))
            .replace("<<ACTION>>", action_str)
            .replace("<<BBOX>>", f"The current bounding box is located at {str(clamped_bbox)} (relative to the cropped image)")
        )
        
        ctx.debug['refine_loop_prompt'] = system_prompt+'\n'+user_prompt
        
        raw_resp, resp = ctx.agent.call_vlm(ctx.vlm, user_prompt=user_prompt, image_path=adjust_img_path, system_prompt=system_prompt)
        
        ctx.logger.info(
            "[INFO] BBOX REFINEMENT LOOP raw response:\n%s\n",
            raw_resp
        )
        
        result = bbox_governor.govern(resp).fixed_output
        
        if result["match"] == "yes":
            
            ctx.logger.info("[INFO] bbox matched. image: %s\n", adjust_img_path)
            
            ctx.logger.info(
                "%s\n[BBOX REFINEMENT-%d]\nnode=%s\nimage=%s\n%s\n%s\n",
                "-" * 60,
                adjust_idx,
                ctx.current_node,
                adjust_img_path,
                result,
                "-" * 60,
            )
            
            padded_bbox = eu.ret_padded_bbox(ctx.driver, bbox_view)
            
            ctx.debug['padded_bbox'] = padded_bbox
            
            eu.save_window_bbox_png(ctx.driver, padded_bbox, f"{crop_img_path}-{adjust_idx}_cropped.png")
            
            bbox_click = eu.convert_bbox_to_crop_space(bbox_view, padded_bbox)
            
            cropped_img = Image.open(f"{crop_img_path}-{adjust_idx}_cropped.png")
            width, height = cropped_img.size
            image_size = {"width": width, "height": height}
            clamped_bbox = eu.clamp_bbox_to_image(bbox_click, image_size)
            
            with open("./prompt/5_ClickPoint_system.md", "r", encoding="utf-8") as f:
                system_prompt = f.read()

            with open("./prompt/5_ClickPoint_user.md", "r", encoding="utf-8") as f:
                user_prompt_template = f.read()
                
            user_prompt = (
                user_prompt_template
                .replace("<<IMAGE_SIZE>>", str(image_size))
                .replace("<<ACTION>>", action_str)
                .replace("<<BBOX>>", f"The current bounding box is located at {str(clamped_bbox)} (relative to the cropped image)")
            )
            
            ctx.debug['prompt_click'] = system_prompt+'\n'+user_prompt
            
            raw_resp, resp = ctx.agent.call_vlm(ctx.vlm, user_prompt=user_prompt, image_path=f"{crop_img_path}-{adjust_idx}_cropped.png", system_prompt=system_prompt)
            
            ctx.logger.info(
                "[INFO] CLICK POINT raw response:\n%s\n",
                raw_resp
            )
            
            result = click_governor.govern(resp).fixed_output
            
            ctx.logger.info(
                "%s\n[GET CLICK POINT-%d]\nnode=%s\nimage=%s\n%s\n%s\n",
                "-" * 60,
                adjust_idx,
                ctx.current_node,
                f"{crop_img_path}-{adjust_idx}-clicked.png",
                result,
                "-" * 60,
            )
            
            click_point_view = eu.convert_crop_point_to_viewport(
                result["click_point"],
                padded_bbox
            )
            eu.show_click_point(ctx.driver, click_point_view)
            eu.save_window_bbox_png(ctx.driver, padded_bbox, f"{crop_img_path}-{adjust_idx}-clicked.png")

            eu.remove_click_point(ctx.driver)
            eu.clear_all_visual_debug(ctx.driver)
            
            eu.execute_action_by_point(ctx.driver, node, click_point_view, value, rank)
            
            ctx.debug['bbox_rel'] = bbox_rel
            ctx.debug['bbox_view'] = bbox_view
            ctx.debug['bbox_click'] = bbox_click
            ctx.debug['click_point_rel'] = result["click_point"]
            ctx.debug['click_point_view'] = click_point_view
            
            return RefineBBoxResult(matched=True, need_prune=False,
                                bbox_rel=None, bbox_view=None,
                                click_point_rel=result["click_point"], click_point_view=click_point_view)
        else:
            ctx.logger.info("[INFO] bbox not matched yet\n")
            ctx.logger.info("[INFO] bbox before padding %d: %s\n", padding, str(result["bbox"]))
            
            if result["bbox"] != None:
                x1, y1, x2, y2 =  result["bbox"]
                bbox_rel = [
                    x1 - padding,
                    y1 - padding,
                    x2 + padding,
                    y2 + padding,
                ]
            else:
                ctx.logger.info("[INFO] Refined bbox is not returned. Let's go back to the initial responded bbox.\n")
                
                x1, y1, x2, y2 = node.action["bbox"]
                bbox_view = [
                    x1 - padding,
                    y1 - padding,
                    x2 + padding,
                    y2 + padding,
                ]
                quadrant_bbox = eu.ret_quadrant_bbox(ctx.driver, bbox_view)
                bbox_rel = eu.convert_bbox_to_crop_space(bbox_view, quadrant_bbox)

            ctx.logger.info(
                "%s\n[BBOX REFINEMENT-%d]\nnode=%s\nimage=%s\n%s\n%s\n",
                "-" * 60,
                adjust_idx,
                ctx.current_node,
                adjust_img_path,
                result,
                "-" * 60,
            )
        
        adjust_idx += 1

    ctx.logger.info("[INFO] Element not found until max trial\n" )
    
    eu.clear_all_visual_debug(ctx.driver)
    
    if result["bbox"] is not None:
        bbox_view = eu.convert_crop_bbox_to_viewport(result["bbox"], quadrant_bbox)
    
        ctx.debug['bbox_rel'] = bbox_rel
        ctx.debug['bbox_view'] = bbox_view
    ctx.debug['click_point_rel'] = None
    ctx.debug['click_point_rel'] = None
            
    return RefineBBoxResult(matched=False,
                            need_prune=True,
                                bbox_rel=result["bbox"], bbox_view=bbox_view,
                                click_point_rel=None, click_point_view=None)

In [ ]:
import components.element_utils
reload(components.element_utils)
from components.element_utils import resolve_action_element_from_item

import components.element_utils
reload(components.element_utils)
from components.element_utils import get_element_bbox

from selenium.webdriver import ActionChains

import time

def prune_and_replay_next(ctx, prune_reason: str, domain):
        
    ctx.tree.prune_node(ctx.current_node, prune_reason)
    
    equiv_nodes = ctx.tree.find_equiv_node_ids(ctx.current_node)
    for en in equiv_nodes:
        ctx.tree.prune_node(en, "equiv node pruned")
        ctx.logger.info(
            "[INFO] pruned node%d: equiv node pruned\n",
            en
        )
    ctx.logger.info(
        "[INFO] pruned current node %d: %s\n",
        ctx.current_node, prune_reason)
    
    next_node = ctx.tree.pick_next_to_expand(ctx.current_node)
    if next_node:
        ctx.current_node = next_node.id
        ctx.logger.info("[INFO] next selected node is %d\n", ctx.current_node)
    else:
        ctx.logger.info("[ERROR] No candidate to expand\n")
        return False
    
    replay = ctx.tree.get_replay_actions(ctx.current_node)
    ctx.driver.get(replay[0])
    
    time.sleep(5)
    ctx.driver.save_screenshot(f"./tmp.png")
    
    state_result = run_state_module(ctx, "./tmp.png")
    if state_result.state == 6:
        ctx.logger.info(
                "[INFO] CAPTCHA detected.\n"
            )
        input("CAPTCHA done? Enter any key to proceed")

    for action in replay[1]:
        ctx.logger.info("[DEBUG] %s\n",str(action))
        ctx.logger.info(
            "[INFO] replaying action: %s\nelement:%s\n",
            action['action'], action['el'])
        
        if action['action'] == "scroll":
            eu.scroll(ctx.driver)
            
            ctx.logger.info(
                "[INFO] scroll executed\n"
            ) 
            time.sleep(5)
        elif action['action'] == "CAPTCHA":
            input("CAPTCHA Done? Enter any key to proceed")
        elif action['action'] == 'navigate':
            ctx.driver.get(action['value'])
            time.sleep(5)
        else:
            if action["el"] is None:
                ctx.logger.info("[INFO] el is None\n")
                continue
            
            if action['state'] == "element":
                element = resolve_action_element_from_item(ctx.driver, action)
                click_point = eu.get_click_point_from_element(ctx.driver, element)
                
                ctx.driver.switch_to.default_content()
            elif action['state'] == "point":
                click_point = action["el"]
            
            value = action['value'] if 'value' in action else None
            eu.execute_action_by_point_replay(ctx.driver, action['action'], click_point, value, domain) 

    ctx.logger.info(
        "[INFO] replay done and continue on %d\n",
        ctx.current_node)
    
    return True

### Initialization

In [ ]:
from datetime import datetime
today = datetime.now().strftime("%y%m%d")
reload(eu)
reload(agent)

from datetime import datetime

domain = "forbes"
home_url = "https://www.forbes.com/"
rank = "199"

rank_domain_date = f"{rank}_{domain}_{today}"

TEMPERATURE = 0 

PROVIDER = "vllm"
MODEL = "Qwen/Qwen3.5-27B-FP8"
BASE_URL = "http://127.0.0.1:30000/v1"
API_KEY = "EMPTY"

PROVIDER = "openai"
MODEL = "gpt-5.1"
BASE_URL = None
API_KEY = "YOUR OPENAI API KEY HERE"

model_str = MODEL.replace("/","_")
screenshot_path = f'./screenshot/{rank_domain_date}_{model_str}'
bbox_dir = f'./screenshot/{rank_domain_date}_{model_str}/debug'

tree_dir = f'./output/tree/{rank}_{domain}'
tree_path = f'{tree_dir}/{rank_domain_date}_{model_str}.json'

log_dir = f'./output/{rank}_{domain}'
log_path = f'{log_dir}/{rank_domain_date}_{model_str}.log'

os.makedirs(screenshot_path, exist_ok=True)
os.makedirs(tree_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)
os.makedirs(bbox_dir, exist_ok=True)

print("screenshot_path:", screenshot_path)
print("bbox_dir:", bbox_dir)
print("tree_dir:", tree_dir)
print("tree_path:", tree_path)
print("log_dir:", log_dir)
print("log_path:", log_path)

import logging
import time
import components.agent as agent

from importlib import reload
import components.ToT as ToT
ToT = reload(ToT)

logging.basicConfig(
    filename=log_path,
    level=logging.INFO,
    force=True
)
logger = logging.getLogger()
logging.getLogger("httpx").disabled = True

tree = ToT.ToTTree()

agent = reload(agent)
vlm = agent.VLMClient(PROVIDER, MODEL, BASE_URL, API_KEY, TEMPERATURE)

PRUNE_STATE = False

ctx = ExecContext(
        vlm=vlm,
        tree=tree,
        agent=agent,
        driver=None ,
        current_node=-1,
        image_path="",
        logger=logger,
        domain=domain
    )

### Run

In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

CLOAK_CHROMEDRIVER = "YOUR CHROMEDRIVER PATH HERE"
CLOAK_BINARY = "YOUR CLOACK BROWSER BINARY PATH HERE"

async def start_browser():
    p = await async_playwright().__aenter__()
    browser = await p.chromium.launch(
        executable_path=CLOAK_BINARY,
        headless=False,
        args=["--remote-debugging-port=9333", "--no-sandbox", "--disable-dev-shm-usage", "--disable-gpu", "--disable-software-rasterizer"]
    )
    page = await browser.new_page(viewport={"width": 1200, "height": 900})
    await page.goto(home_url)
    
    return p, browser, page

pw, browser, page = asyncio.run(start_browser())
options = Options()
options.add_experimental_option("debuggerAddress", "127.0.0.1:9333")
driver = webdriver.Chrome(
    options=options,
    service=Service(CLOAK_CHROMEDRIVER)
)
print(driver.current_url)

print("screenshot_path:", screenshot_path)
print("log_path:", log_path)

In [ ]:
import asyncio, nest_asyncio
nest_asyncio.apply()

from playwright.async_api import async_playwright
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service

ctx.driver = driver

new_node_id = ctx.tree.new_node_id()
ctx.tree.add_root(node_id=new_node_id, href=driver.current_url)
ctx.current_node = new_node_id
ctx.driver.save_screenshot(f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png")

PRUNE_STATE = False

actionchain = ActionChains(ctx.driver)

for i in range(50):

    print("current node:", ctx.current_node)
    print(PRUNE_STATE)
    
    ############################STATE MODULE########################

    if PRUNE_STATE == False:
    
        ctx.image_path = f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png"
        
        state_result = run_state_module(ctx)

        if state_result.decision == "prune":
            PRUNE_STATE = True
            
            if not prune_and_replay_next(ctx, "decision==prune", domain):
                raise RuntimeError("No more next node")

        if state_result.state == 6:
            ctx.logger.info(
                "[INFO] CAPTCHA detected.\n"
            )
            
            captcha_action = {
                "action_type": "CAPTCHA",
                "kind": "CAPTCHA",
                "text_hint": None,
                "icon_hint": None,
                "bbox": None,
                "rationale": "detected by state identifier"
            }
            nid = ctx.tree.new_node_id()
            ctx.tree.mark_expanded(ctx.current_node)
            
            ctx.tree.add_child(
                node_id=nid,
                parent_id=ctx.current_node,
                action=captcha_action,
                confidence_score=10,
                state=6,
                href = driver.current_url
            )
            
            ctx.current_node = nid
            
            element_info = dict()
            element_info["action"] = "CAPTCHA"
            element_info["el"] = None
            element_info["value"] = "CAPTCHA_solver"
            ctx.tree.update_element(ctx.current_node, element_info)
            
            ctx.driver.save_screenshot(f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png")
            
            continue
            
        if state_result.state == 5:
            print("=========Automation DONE============")
            
            ctx.tree.mark_expanded(ctx.current_node)
            
            ctx.logger.info(
            "=========Automation DONE============")
            ctx.logger.info(json.dumps(ctx.tree.get_pwd_actions(ctx.current_node), ensure_ascii=False))
            break

        ctx.state = state_result.state
        
        keywords = state_result.keywords
        
        if ctx.state == 3:
            targets = ["user", "profile", "menu", "more", "settings"]
            existing = {item["text"].lower() for item in keywords}

            for t in targets:
                if t.lower() not in existing:
                    keywords.append({"text": t, "weight": 1})
        elif ctx.state == 1:
            targets = ["user", "profile"]
            existing = {item["text"].lower() for item in keywords}

            for t in targets:
                if t.lower() not in existing:
                    keywords.append({"text": t, "weight": 1})
    
    #############################ACTION MODULE########################
    
    if PRUNE_STATE == False:
        action_result = run_action_module(
            ctx,
            keywords,
            action_governor
            # log_path,
        )

        if action_result.next_node is None:
            if not prune_and_replay_next(ctx, "No action generated", domain):
                raise RuntimeError("No more next node")
        else:
            ctx.current_node = action_result.next_node
        print(ctx.current_node)
    
    else:
        PRUNE_STATE = False
    
    ############################ELEMENT MODULE########################
    
    node = ctx.tree.nodes[ctx.current_node]
    
    element_info = dict()
    element_info["action"] = node.action["action_type"]
    element_info["el"] = None
    element_info["value"] = None
    
    if node.action['action_type'] == "scroll":
        
        ctx.tree.update_element(ctx.current_node, element_info)
        
        at_bottom = eu.chk_bottom(ctx.driver)
        
        if not at_bottom:
 
            eu.scroll(ctx.driver)
            
            print('scroll')
            ctx.logger.info(
                "[INFO] scroll executed\n"
            ) 
            
            time.sleep(5)
            ctx.tree.update_href(ctx.current_node, ctx.driver.current_url)
            ctx.driver.save_screenshot(f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png")
            ctx.tree.save_tree(tree_path)
            
            if eu.chk_bottom(ctx.driver):
                ctx.scrolled.append(ctx.driver.current_url)
            
            continue
        
        else:
            PRUNE_STATE = True
            
            ctx.scrolled.append(ctx.driver.current_url)
            
            print('reached bottom')            
            if not prune_and_replay_next(ctx, "scroll reached bottom", domain):
                raise RuntimeError("No more next node")

            continue
    
    elif node.action['action_type'] == "navigate":
        
        ctx.tree.update_element(ctx.current_node, element_info)
        
        href = node.action["text_hint"]
        element_info["value"] = href
        ctx.driver.get(href)
        
        ctx.logger.info(
                "[INFO] Navigate to %s executed\n", href
            ) 
        
        time.sleep(5)
        ctx.tree.update_href(ctx.current_node, ctx.driver.current_url)
        ctx.driver.save_screenshot(f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png")
        ctx.tree.save_tree(tree_path)
        
        continue
    
    elif node.action['action_type'] == 'wait':
        
        ctx.tree.update_element(ctx.current_node, element_info)
        
        time.sleep(5)
        
        ctx.tree.update_href(ctx.current_node, ctx.driver.current_url)
        ctx.driver.save_screenshot(f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png")
        ctx.tree.save_tree(tree_path)
        
        continue
    
    keyword_items = action_result.items
    print(keyword_items)
    keyword_item = eu.find_matching_item(node.action, keyword_items)
    
    if keyword_item is not None and node.action['kind'] != 'input':
        print('here keyword_item', keyword_item)
        ctx.logger.info("[INFO] keyword_element: %s", str(keyword_item))
        
        element_info = keyword_item
        # element_info["el"] = None
        element_info['state'] = "element"
        
        if (
            keyword_item['tag'] == 'a'
            and eu.is_navigable_href((keyword_item.get('attrs') or {}).get('href', ''))
            and (keyword_item.get('attrs') or {}).get('type') != 'button'
            ):
            # node.action['action_type'] = 'navigate'
            
            current_url = ctx.driver.execute_script("""return document.querySelector('base')?.href || window.location.href;""")
            href_str = urljoin(current_url, keyword_item['attrs']['href'])
            # node.action['text_hint'] = href_str
            element_info['value'] = href_str
            element_info['state'] = "navigate"
            if (ctx.state == 1 or ctx.state==2) and MODE=="manual":
                input("ACTION DONE?")
            else:
                ctx.driver.get(href_str)
            
            time.sleep(5)
        else:
            print('here else')
            print(keyword_item['bbox'])
            node.action['action_type'] = 'click'
            element_bbox = keyword_item['bbox']
            click_point = eu.get_click_point_from_bbox(element_bbox)
            
            ctx.logger.info("[INFO] element_bbox: %s, click_point: %s\n", element_bbox,click_point)
            
            print(element_bbox, click_point)
            
            if (ctx.state == 1 or ctx.state==2) and MODE=="manual":
                input("ACTION DONE?")
            else:
                eu.execute_action_by_point(ctx.driver, node, click_point, 'null', domain,ctx.debug['action_result'].fixed_output)
        
        element_info["el"] = None
        element_info['action'] = node.action['action_type']
        
        ctx.tree.update_element(ctx.current_node, element_info)
    
    else:
        # ---------- ELEMENT MODULE (selection only) ----------
        NEED_ADJUST = False
        
        elem_res = run_element_module(
            ctx,
            bbox_dir=bbox_dir,
            element_governor=element_governor,
            CHUNK_SIZE=100,
        )
        
        relevance = elem_res.relevance
        quadrant_bbox = elem_res.quadrant_bbox
        value = elem_res.value
        
        #############################EXECUTION########################

        hit_index = elem_res.selected_index  # 1-based
        
        crop_img_path = f"{screenshot_path}/debug/{ctx.current_node}"
        
        if hit_index > 0:
            filtered_elements = elem_res.filtered_elements
            raw_htmls = elem_res.raw_htmls

            item = filtered_elements[hit_index - 1]
            
            element_info = item
            element_info["action"] = node.action["action_type"]
            element_info['state'] = "element"
            
        else:
            NEED_ADJUST = True
            
            element_info["action"] = node.action["action_type"]
            element_info['state'] = "point"
            
            eu.save_window_bbox_png(driver, quadrant_bbox, crop_img_path+".png")
        
        element_info["value"] = elem_res.value  
        
        ctx.tree.update_relevance(ctx.current_node, relevance)
        
        if value == 'CAPTCHA_solver' or value == 'human_intervention':
            ctx.logger.info(
                "[INFO]%s activated\n",
                value
            )
            input("Enter any key to proceed")
        else:
            action_wo_bbox = {k: v for k, v in node.action.items() if k != "bbox"}
            action_str = json.dumps(action_wo_bbox, indent=2, ensure_ascii=False)    
                
            if hit_index > 0 :
                element, is_shadow = resolve_action_element(ctx.driver, item)
                
                element_bbox = eu.get_element_bbox(ctx.driver, item)
                quadrant_bbox = eu.ret_quadrant_bbox(ctx.driver, element_bbox)
                ctx.debug['quadrant_bbox'] = quadrant_bbox
                
                element_info["el"] = raw_htmls[hit_index - 1]
                
                refine_res = run_refine_bbox_module(ctx, element_bbox, quadrant_bbox, action_str, crop_img_path, padding=10)
                
                # break
                if refine_res.matched==False:
                    NEED_ADJUST = True
                    bbox_rel = refine_res.bbox_rel
                    bbox_view = refine_res.bbox_view
            
            if NEED_ADJUST == True:
                refine_loop_res = run_bbox_adjustment_loop(ctx, node, quadrant_bbox, crop_img_path, action_str, value, rank, padding=10)
                
                if refine_loop_res.need_prune == True:
                    PRUNE_STATE = True
                    if not prune_and_replay_next(ctx, "refine bbox failed", domain):
                        raise RuntimeError("No more next node")
                    continue
                else:
                    element_info['el'] = refine_res.click_point_view
            
            ############ VERIFY ELEMENT ################

            else:  
                
                click_point = eu.get_click_point_from_bbox(element_bbox)
                if (ctx.state == 1 or ctx.state==2) and MODE=="manual":
                    input("ACTION DONE?")
                else:
                    ctx.logger.info("[INFO] element_bbox: %s, click_point: %s\n", element_bbox,click_point)
                    
                    eu.execute_action_by_point(ctx.driver, node, click_point, value, domain, ctx.debug['action_result'].fixed_output) 

    ctx.tree.update_element(ctx.current_node, element_info)
    
    ctx.tree.update_href(ctx.current_node, ctx.driver.current_url)
    
    ctx.driver.save_screenshot(
        f"{screenshot_path}/{rank}_{domain}-{ctx.current_node}.png"
    )
    ctx.tree.save_tree(tree_path)
    
    action_trace = str()
    for idx, action in enumerate(node.path_actions, 1):
        action_wo_bbox = {k: v for k, v in action.items() if k != "bbox"}
        action_trace += f"{idx}. {json.dumps(action_wo_bbox, ensure_ascii=False)}\n"
    if action_trace == "":
    
        action_trace = "NULL"
        
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(action_trace)
        f.write("================================================\n\n")


### Visualize Tree

In [ ]:
reload(ToT)
from components.ToT import visualize_tot_tree

tree = ctx.tree

visualize_tot_tree(tree, tree_dir, rank, domain, today, model_str,figsize=(20, 20))